# Predicting Kickstarter Project Success


In this notebook we build a **binary classification** model that predicts
whether a Kickstarter project will be **successful** or **failed**. We
will:

1. Load a preprocessed sample of the Kickstarter dataset.
2. Engineer features from raw columns (dates, videos, categories).
3. Train a gradient-boosted tree model with **XGBoost**.
4. Evaluate it with accuracy, ROC-AUC, precision, recall, and F1, and
   discuss how those metrics are used in real-life analytics work.

:::{important} About the data and the results

This notebook uses a **small sample** of the full Kickstarter dataset for
fast iteration in class. Models trained on a small, narrow slice of data
often look much better on their held-out test set than they would on the
full population. As you read the results, treat the headline numbers as
*illustrative* rather than as a realistic estimate of production
performance. We will revisit these results on the larger dataset in a
later chapter.

:::


In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb

import sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

print(f"pandas version: {pd.__version__}")
print(f"numpy version: {np.__version__}")
print(f"xgboost version: {xgb.__version__}")
print(f"scikit-learn version: {sklearn.__version__}")

pandas version: 3.0.0
numpy version: 2.4.2
xgboost version: 3.2.0
scikit-learn version: 1.8.0


Pandas does not ship with a Parquet engine.To read a Parquet file as a Pandas DataFrame, you must install one of the Parquet engines (`pyarrow` or `fastparquet`) and specify the engine.

**Using pip**

```bash
pip install pyarrow
```

**Using conda**

```bash
conda install -c conda-forge pyarrow
```


In [2]:
# Uncomment and run the line below to install pyarrow if you haven't already
# %pip install pyarrow

## Load the Data

We work with a small, pre-cleaned subset of the Kickstarter dataset stored
as a Parquet file. Parquet is a columnar binary format that is much faster
to read than CSV and preserves data types (so date columns stay as
datetimes, etc.).

After loading, we use `df.head()` and `df.info()` to get a feel for the
columns, types, and non-null counts. This is always a good first move on
a new dataset.


In [3]:
df = pd.read_parquet(
    "https://github.com/bdi593/datasets/raw/refs/heads/main/kickstarter-projects/kickstarter-sample-preprocessed.parquet"
)
df.head()

,name,state,backers_count,usd_pledged,goal,percent_funded,launched_at,state_changed_at,country,currency,staff_pick,spotlight,category,category_parent,video,blurb,video_width,video_height
0,Stained Glass Mosaics,failed,4,13.000000,1500,0.866667,2016-07-19 22:35:04,2016-08-18 22:35:04,US,USD,False,False,Textiles,Art,NaN,Original works of art created with strong pass...,NaN,NaN
1,Personalized Boutique Balloons,failed,1,7.248741,1000,1.000000,2025-12-21 19:14:50,2026-01-31 23:50:00,CA,CAD,False,False,DIY,Crafts,NaN,You are the focus of luxury crafted arrangemen...,NaN,NaN
2,From Pop-Up to Permanent,failed,2,316.000000,4500,7.022222,2025-12-11 05:42:21,2026-01-15 05:42:21,US,USD,False,False,Candles,Crafts,NaN,Join us in building a studio home for Raven E'...,NaN,NaN
3,"43 Amp Arduino Motor shield, the Arc-Controller",failed,28,2565.000000,41500,6.180723,2014-08-14 06:02:45,2014-09-28 06:02:45,US,USD,False,False,DIY Electronics,Technology,"{""id"":426572,""status"":""successful"",""hls"":null,...",The Arc-Controller is here to give power to th...,640.0,360.0
4,The SnapPiCam | A Raspberry Pi Digital Camera.,failed,84,18503.697893,40000,27.340000,2014-08-01 09:41:01,2014-08-31 09:41:03,GB,GBP,True,False,DIY Electronics,Technology,NaN,Build your own Raspberry Pi powered Touchscree...,NaN,NaN


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2706 entries, 0 to 2705
Data columns (total 18 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   name              2706 non-null   str           
 1   state             2706 non-null   str           
 2   backers_count     2706 non-null   int64         
 3   usd_pledged       2706 non-null   float64       
 4   goal              2706 non-null   int64         
 5   percent_funded    2706 non-null   float64       
 6   launched_at       2706 non-null   datetime64[us]
 7   state_changed_at  2706 non-null   datetime64[us]
 8   country           2706 non-null   str           
 9   currency          2706 non-null   str           
 10  staff_pick        2706 non-null   bool          
 11  spotlight         2706 non-null   bool          
 12  category          2706 non-null   str           
 13  category_parent   2654 non-null   str           
 14  video             1918 non-null   s

## Preprocess the Data for Binary Classification

Raw data is almost never in a shape a model can consume directly. In this
section we will:

1. Define a clean **target column** (success vs. failure).
2. Engineer a few **date features** from the launch timestamp.
3. Add a simple **video indicator**.
4. Clean up missing values in the **category** column.
5. Decide which **features** to include — and explicitly call out the ones
   that would cause data leakage.

Each of these steps mirrors what an analyst would do on a real project.


### Target Column

For binary classification we need a single column with two values: 1 for
the positive class and 0 for the negative class. The raw `state` column has
several possible values (such as `successful`, `failed`, `canceled`, and
`live`), so we collapse it into a clean 0/1 indicator.

We treat `successful` as the positive class (1) and everything else as the
negative class (0). Before doing the conversion we look at
`value_counts()` to confirm the class balance — a quick sanity check that
keeps us from accidentally training on, say, a column that is 99% one
class.


In [5]:
df["state"].value_counts()

state
successful    1676
failed        1030
Name: count, dtype: int64

In [6]:
df["state"].value_counts(normalize=True)

state
successful    0.619364
failed        0.380636
Name: proportion, dtype: float64

In [7]:
df["target"] = (df["state"] == "successful").astype(np.int64)

### Date Features

Models cannot consume raw timestamps directly — `2017-04-12 09:31:00` is
not a number the algorithm can compare. Instead, we decompose the launch
timestamp into a few interpretable components:

- **`launch_year`** captures slow-moving trends (the platform is more
  crowded in 2020 than in 2012).
- **`launch_month`** captures seasonality (e.g., a possible holiday-season
  boost).
- **`launch_dayofweek`** captures weekly patterns (campaigns launched on a
  Tuesday may behave differently from those launched on a Saturday).

This is a textbook example of **feature engineering**: turning raw data
into columns that an algorithm can use.


In [8]:
df["launch_year"] = df["launched_at"].dt.year
df["launch_month"] = df["launched_at"].dt.month
df["launch_dayofweek"] = df["launched_at"].dt.dayofweek

### Video Feature

The `video` column stores nested information about each project's pitch
video, but for this first model we only care whether a video exists at all.
We engineer a simple binary `has_video` indicator: a common pattern in real
analytics work where the *presence* of an asset (a photo, a phone number,
a verified email) is itself a strong signal even if we have not parsed the
content yet.


In [9]:
df["has_video"] = df["video"].notna().astype(np.int64)

### Category

Categorical columns like `category` and `category_parent` describe the type
of project (for example *Tabletop Games*, *Indie Rock*, *Documentary*).
They are often the strongest predictors in crowdfunding data because backer
communities are organized around interests, not just price points.


The `"category_parent"` column contains the parent category of the project. The column contains missing (null) values, which we will fill with the string `"Unknown"`.


In [10]:
df["category_parent"] = df["category_parent"].fillna("Unknown")

### Feature Selection

Feature selection is the step where we decide which columns the model is
allowed to look at. There are three competing pressures:

1. **Predictive power** — we want features that actually carry signal about
   the outcome.
2. **Availability at prediction time** — a feature is only useful if we can
   compute it *before* we know the answer (otherwise we have data leakage).
3. **Stability and cost** — features that are noisy, expensive to compute,
   or change definitions over time tend to cause problems in production.

The list below is a deliberate, conservative starting set. We will discuss
the trade-offs around `backers_count` in the admonition that follows.


In [11]:
features = [
    "goal",
    "backers_count",  # ⚠️ Read the note on data leakage below!
    "country",
    "currency",
    "staff_pick",
    "category",
    "category_parent",
    "has_video",
    "launch_year",
    "launch_month",
    "launch_dayofweek",
]

:::{attention} Watch out for data leakage!

Data leakage occurs when the training data contains information that would not be available at prediction time. This can lead to overly optimistic performance estimates during model evaluation, but poor performance on unseen data.

In this case, the `"backers_count"` column contains the number of backers at the time of prediction, not the final number of backers. We are still going to include this feature in our model, but be aware that the timing of the prediction will affect the number of backers. For example, if the project only has been live for a day, it may have very few backers, but if it has been live for a month, it may have many backers.

:::


:::{note} Other excluded features

These features are not included in our model because they contain information that would not be final at the time of prediction.

- `usd_pledged`
- `percent_funded`
- `state_changed_at`
- `spotlight`

The final values are determined after the campaign outcome. You can engineer features from these columns to include them in the model, but we will not do that in this notebook.

:::


### Select Features and Target Columns

By convention we use `X` for the feature matrix (the inputs the model will
look at) and `y` for the target column (what we are trying to predict).
Keeping this separation explicit makes it easy to swap in different feature
sets later when comparing model versions.


In [12]:
X = df[features]
y = df["target"]

### Handle Categoricals

XGBoost can use categorical columns directly when we set
`enable_categorical=True` later, but we first need to convert pandas
`object` (string) columns to the dedicated `category` dtype. This is more
memory efficient and lets the model treat each level as its own group
rather than trying to interpret strings as numbers.


In [13]:
for col in X.select_dtypes(include="object").columns:
    X[col] = X[col].astype("category")

C:\Users\ypark32\AppData\Local\Temp\ipykernel_31892\2429138195.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in X.select_dtypes(include="object").columns:


## Build a Model

With the features prepared, we are ready to train a binary classifier. The
workflow has three steps:

1. Split the data into training and test sets.
2. Fit the model on the training set.
3. Evaluate it on the held-out test set.

Keeping the evaluation set strictly separate from training is the single
most important habit in applied ML — it is how we avoid fooling ourselves
about how well the model really works.


### Split the Data into Training and Test Sets

Before training we set aside a test set the model will *never see* during
training. This gives us an honest estimate of how the model would perform
on brand-new projects.

A few choices to highlight:

- **`test_size=0.2`** — keep 80% of the data for training and 20% for
  evaluation. This is a common starting point; with very small datasets you
  might use cross-validation instead.
- **`random_state=42`** — fixes the shuffling so the split is reproducible.
- **`stratify=y`** — ensures the train and test sets have the *same*
  proportion of successful vs. failed projects as the full dataset. This
  matters whenever the target is even slightly imbalanced; otherwise you
  can end up with a test set whose class mix does not match production.

In a real-life forecasting or fraud setting you would often go further and
use a **time-based split** (train on older projects, test on newer ones) to
mimic how the model will actually be used over time.


In [14]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

### Train the XGBoost Model

We use **XGBoost**, a gradient-boosted decision tree library that is a very
common first choice for tabular classification problems in industry. It
tends to work well out of the box, handles mixed numeric and categorical
features (when `enable_categorical=True`), is robust to missing values, and
trains quickly even on large datasets.

A few defaults worth knowing about:

- `random_state=42` makes the run reproducible.
- `enable_categorical=True` lets us pass the pandas `category` columns we
  created above directly into the model — no manual one-hot encoding needed.
- We are not tuning any hyperparameters in this first pass. In a real
  project, we would later do cross-validated hyperparameter search (for
  example over `max_depth`, `learning_rate`, `n_estimators`, and
  `min_child_weight`) to squeeze out more performance.


In [ ]:
model = xgb.XGBClassifier(
    enable_categorical=True,
    random_state=42,
)

model.fit(X_train, y_train)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes f

## Evaluate the Model

Training accuracy is not a trustworthy measure of how well the model will do
on new projects — a model can memorize its training set. To get an honest
estimate of generalization performance we score the model on the held-out
test set we set aside earlier.

In real-life analytics work, evaluation is rarely a single number. We
typically:

1. Look at multiple metrics so we understand *which kind* of mistake the
   model is making (false positives vs. false negatives).
2. Slice performance by important sub-populations (here: category, country,
   year) to make sure the model is not great on average but terrible for an
   important segment.
3. Compare against a simple baseline (e.g., "always predict the majority
   class") to confirm the model is actually adding value.


### Model Performance

Now that the model is trained, we evaluate it on the held-out test set. We
use the trained model to produce two outputs for every test row:

- `y_pred` — a hard 0/1 prediction (the model's "best guess" class).
- `y_prob` — the predicted probability of the positive class (success).

We need the probabilities to compute ROC-AUC and to experiment with
different decision thresholds later on.


In [16]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_prob))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.966789667896679
ROC AUC: 0.9983385344429034

Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.94      0.96       206
           1       0.96      0.99      0.97       336

    accuracy                           0.97       542
   macro avg       0.97      0.96      0.96       542
weighted avg       0.97      0.97      0.97       542



#### Interpretation of Model Performance Metrics

| Metric       | Value     | Interpretation                   |
| ------------ | --------- | -------------------------------- |
| **Accuracy** | **0.967** | 96.7% of predictions are correct |
| **ROC AUC**  | **0.998** | Near-perfect class separation    |

Accuracy alone can be misleading. We can further evaluate the model using
precision, recall, and F1-score for each class:

| Class              | Support | Precision | Recall | F1-Score |
| ------------------ | ------- | --------- | ------ | -------- |
| **0 (Failed)**     | 206     | 0.97      | 0.94   | 0.96     |
| **1 (Successful)** | 336     | 0.96      | 0.99   | 0.97     |
| **Macro Avg**      | 542     | 0.97      | 0.96   | 0.96     |
| **Weighted Avg**   | 542     | 0.97      | 0.97   | 0.97     |

**What This Means**

- The model rarely misclassifies either class.
- It captures 99% of successful projects.
- It correctly identifies 94% of failures.
- The AUC of 0.998 indicates almost perfect ranking ability.

**What the Metrics Mean**

- **Support**: Number of actual observations in each class (how many true
  failed vs. successful projects in the test set).
- **Precision**: Of the projects predicted as a given class, the proportion
  that were actually correct. *Precision answers: "When the model says yes,
  how often is it right?"*
- **Recall**: Of all true projects in a class, the proportion the model
  successfully identified. *Recall answers: "Of the things we cared about,
  how many did we catch?"*
- **F1-Score**: The harmonic mean of precision and recall — balances false
  positives and false negatives. Useful when classes are imbalanced or when
  both error types are costly.
- **Accuracy**: The fraction of all predictions that were correct. Easy to
  explain, but can be misleading when one class dominates (e.g., a model
  that always predicts "failed" on a 95%-failure dataset would still hit
  95% accuracy).
- **ROC AUC**: Measures how well the model *ranks* successes above failures
  across every possible probability cutoff. A value of 0.5 is random and 1.0
  is perfect. AUC is threshold-independent, so it is useful when you have
  not yet decided what probability cutoff to use in production.

**How these metrics show up in real-life analytics**

The "right" metric depends on the business cost of each kind of mistake.

- *Fraud detection / churn alerts.* The cost of missing a true positive
  (letting a fraudster through, losing a customer silently) is typically
  much higher than the cost of a false alarm. Analytics teams optimize for
  **recall** on the positive class, often accepting lower precision and
  routing flagged cases to a human reviewer.
- *Marketing targeting / sales lead scoring.* Outreach has a per-contact
  cost (emails, sales rep time), so we want most of the people we *do*
  contact to actually convert. That means we optimize for **precision** on
  the positive class.
- *Crowdfunding success prediction (this case).* If we built a "should I
  back this project?" recommender, false positives (recommending a doomed
  project) have a real reputational cost, so we would care about precision.
  If we built a tool that *flagged risky projects for the Kickstarter trust
  & safety team*, false negatives would matter more, so recall would win.
- *Regulated or imbalanced settings.* In medical screening, credit
  decisions, or anti-money-laundering, teams almost always report
  precision, recall, F1, and AUC together — and frequently look at the full
  precision-recall curve at multiple thresholds — because a single number
  cannot capture the trade-off.

A good rule of thumb: **decide which mistake is more expensive *before* you
look at the metrics, then pick the threshold and metric that match that
trade-off.**


### Feature Importance


In [17]:
df_feature_importances = pd.DataFrame(
    {"feature": X_train.columns, "importance": model.feature_importances_}
)

df_feature_importances

,feature,importance
0,goal,0.070064
1,backers_count,0.239073
2,country,0.018279
3,currency,0.024946
4,staff_pick,0.013598
5,category,0.517437
6,category_parent,0.050327
7,has_video,0.006112
8,launch_year,0.039000
9,launch_month,0.008846


#### Interpretation of Feature Importance

Feature importance tells us how much each input contributed to the model's
splits. It is one of the most common ways analysts translate a "black box"
model into something a business stakeholder can act on.

Here is a sample output of the feature importance DataFrame:

| Feature          | Importance | Interpretation                |
| ---------------- | ---------- | ----------------------------- |
| **category**     | **0.517**  | Dominates model decisions     |
| backers_count    | 0.239      | Very strong signal (leakage)  |
| goal             | 0.070      | Moderate signal               |
| category_parent  | 0.050      | Adds additional grouping info |
| launch_year      | 0.039      | Some time trend effect        |
| currency         | 0.025      | Minor                         |
| country          | 0.018      | Minor                         |
| staff_pick       | 0.014      | Small (surprisingly low here) |
| launch_dayofweek | 0.012      | Very small                    |
| launch_month     | 0.009      | Very small                    |
| has_video        | 0.006      | Negligible                    |

**How analysts use this in practice**

- *Validate intuition.* If the most important features match what subject
  matter experts expect, that is a sanity check. If they do not, it is a
  signal to investigate (often a leakage bug, like `backers_count` here).
- *Identify leakage.* A single feature with a suspiciously high importance
  is one of the most reliable warning signs that the feature contains
  information that would not be available at prediction time.
- *Prioritize data collection.* Features that show up at the top are worth
  investing in: better cleaning, more frequent refresh, or richer encodings.
- *Trim noise.* Features at the bottom can often be dropped to simplify the
  model and reduce maintenance cost without losing accuracy.

Keep in mind that XGBoost's default importance is based on how often a
feature is used to split the trees. It does *not* tell us the direction of
the effect (does a high goal increase or decrease success?). For that we
would normally use SHAP values or partial dependence plots.


### Category Analysis


In [18]:
df.groupby(["category", "state"], as_index=False).size()

,category,state,size
0,3D Printing,successful,12
1,Academic,failed,18
2,Accessories,successful,22
3,Action,failed,47
4,Action,successful,3
...,...,...,...
178,World Music,failed,1
179,World Music,successful,33
180,Young Adult,failed,54
181,Zines,failed,32


In [19]:
category_summary = df.groupby("category").agg(
    total_projects=("state", "count"),
    successes=("target", "sum"),
    success_rate=("target", "mean"),
)

category_summary.head(10)

,total_projects,successes,success_rate
category,,,
3D Printing,12,12,1.00
Academic,18,0,0.00
Accessories,22,22,1.00
Action,50,3,0.06
Animals,1,1,1.00
Animation,25,7,0.28
Anthologies,67,67,1.00
Apparel,36,36,1.00
Apps,10,0,0.00


:::{attention} Why does this model look so good?

The accuracy and ROC-AUC reported above are unusually high for a real-world
predictive modeling problem. Before celebrating, it is worth pausing to ask
*why*.

1. **The dataset is a tiny subset of the full Kickstarter data.** We are
   training and testing on a sample of only a few thousand projects, drawn
   from a very specific slice of time. Real Kickstarter data contains
   millions of projects across many years and categories, with much more
   variety in goal sizes, currencies, and creator behavior. Models trained
   on small, narrow samples often look great on their own test split but
   perform noticeably worse when applied to the full population.
2. **`backers_count` leaks information about the outcome.** As called out
   earlier, the number of backers is heavily correlated with success. In a
   real production setting we usually do not know the *final* backer count
   at the moment we want to make a prediction. Removing or time-bounding
   this feature typically drops accuracy by a wide margin.
3. **Category imbalance can mask weak performance.** Although the overall
   distribution of successful vs. failed projects is roughly balanced, the
   success rate inside each category varies a lot. A model can score well
   overall by simply memorizing which categories tend to succeed in this
   particular sample.

In the upcoming chapters we will work with the larger preprocessed dataset
to see whether this strong performance holds up, and we will revisit
feature engineering choices that avoid leakage.

:::
